In [1]:
import pandas as pd

orders_full = pd.read_csv('data_processed/orders_full_clean.csv')
orders_full.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'delivery_delay_days']

In [6]:
# Recreate on_time flag (same logic as notebook 02)
orders_full['on_time'] = orders_full['delivery_delay_days'] <= 0

# Bring in review scores (same as notebook 02)
order_reviews = pd.read_csv('archive-2/olist_order_reviews_dataset.csv')
review_scores = order_reviews.groupby('order_id')['review_score'].mean().reset_index()
orders_full = orders_full.merge(review_scores, on='order_id', how='left')

orders_full[['on_time', 'review_score']].info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   on_time       113425 non-null  bool   
 1   review_score  112464 non-null  float64
dtypes: bool(1), float64(1)
memory usage: 997.0 KB


In [7]:
seller_stats = orders_full.groupby('seller_id').agg(
    total_orders=('order_id', 'count'),
    pct_late=('on_time', lambda x: 1 - x.mean()),
    avg_review_score=('review_score', 'mean')
).reset_index()

seller_stats.head()

,seller_id,total_orders,pct_late,avg_review_score
0,0015a82c2db000af6aaaf3ae2ecb0532,3,0.000000,3.666667
1,001cca7ae9ae17fb1caed9dfb1094831,239,0.071130,3.902542
2,001e6ad469a905060d959994f1b41e4f,1,1.000000,1.000000
3,002100f778ceb8431b7a1020ff7ab48f,55,0.181818,3.981818
4,003554e2dce176b5555353e4f3555ac8,1,0.000000,5.000000


In [9]:
# Only look at sellers with a meaningful order history
reliable_sellers = seller_stats[seller_stats['total_orders'] >= 5].copy()

# Risk score: weighted combination of late % and low reviews
# Higher score = riskier seller
reliable_sellers['risk_score'] = (
    reliable_sellers['pct_late'] * 0.6 +
    (1 - reliable_sellers['avg_review_score'] / 5) * 0.4
)

# Tier sellers into Low / Medium / High risk
reliable_sellers['risk_tier'] = pd.cut(
    reliable_sellers['risk_score'],
    bins=[-0.01, 0.15, 0.35, 1.0],
    labels=['Low', 'Medium', 'High']
)

reliable_sellers.sort_values('risk_score', ascending=False).head(10)

,seller_id,total_orders,pct_late,avg_review_score,risk_score,risk_tier
1777,90d4125885ab6c86e8820a722be71974,5,1.000000,1.000000,0.920000,High
1952,a0e19590a0923cdd0614ea9427713ced,7,1.000000,1.000000,0.920000,High
705,3bfad056cf05c00dabe2f895925d83b1,5,1.000000,1.000000,0.920000,High
809,4342d4b2ba6b161468c63a7e7cfce593,20,0.950000,1.263158,0.868947,High
1734,8d92f3ea807b89465643c219455e7369,8,0.875000,1.000000,0.845000,High
1227,64b87978a11c1ea7501a89eabe8c2c1a,7,1.000000,2.000000,0.840000,High
1720,8c3b533c63cca56240f94f1e3a6b18ef,8,0.875000,1.333333,0.818333,High
938,4e2627090e6e5b9fabba883a37897683,5,0.800000,1.000000,0.800000,High
1493,7994081388d55b9cf976c050653dd359,6,0.833333,1.666667,0.766667,High
1355,6ee85be3693ed79a8e80718743d80655,13,0.769231,1.923077,0.707692,High


In [10]:
reliable_sellers['risk_tier'].value_counts()

risk_tier
Low       1337
Medium     476
High        81
Name: count, dtype: int64

In [11]:
reliable_sellers.sort_values('risk_score', ascending=False).to_csv('data_processed/seller_risk_scores.csv', index=False)
print("Saved seller risk scores to data_processed/seller_risk_scores.csv")

Saved seller risk scores to data_processed/seller_risk_scores.csv


## Key Findings — Seller Risk Analysis

- Built a seller-level risk score combining late delivery rate (60% weight) and average review score (40% weight), based on the finding from notebook 02 that delivery delay is the strongest driver of poor reviews.
- Analysis restricted to sellers with 5+ orders (1,894 sellers) to avoid unreliable scores from low-volume sellers.
- **Results: 1,337 sellers (70.6%) Low risk, 476 (25.1%) Medium risk, 81 (4.3%) High risk.**
- High-risk sellers show consistent patterns of near-100% late delivery combined with average review scores near 1.0-2.0, even at meaningful order volumes (10-20+ orders) — indicating a real seller-level problem rather than one-off bad luck.

This risk scoring framework could be used by Olist's operations team to proactively flag underperforming sellers for review, outreach, or removal from the platform before they generate a high volume of poor customer experiences.